In [34]:
import tensorflow as tf
import numpy as np
from tensorflow.keras import layers, models

def build_lightweight_cnn(input_shape=(24, 1)):
    model = models.Sequential([
        layers.Conv1D(32, kernel_size=3, padding='same', input_shape=input_shape),
        layers.BatchNormalization(), # <--- Keeps the math from drifting
        layers.Activation('relu'),
        layers.MaxPooling1D(2),
        
        layers.Conv1D(64, kernel_size=3, padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.GlobalAveragePooling1D(),
        
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.3), 
        layers.Dense(1, activation='sigmoid') 
    ])

    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

model = build_lightweight_cnn()
model.summary()


C:\Users\percy\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_13"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_26 (Conv1D)              │ (None, 24, 32)         │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 24, 32)         │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_6 (Activation)       │ (None, 24, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_13 (MaxPooling1D) │ (None, 12, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_27 (Conv1D)              │ (None, 12, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 12, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_7 (Activation)       │ (None, 12, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_13     │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,833 (34.50 KB)

 Trainable params: 8,641 (33.75 KB)

 Non-trainable params: 192 (768.00 B)

In [46]:
import pandas as pd
from sklearn.discriminant_analysis import StandardScaler
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
import joblib


input = pd.read_csv("acoustic_features_24.csv")
input = input.dropna(inplace=False)
X = input.drop(columns=["label"]).values
y = input["label"].values

scaler = StandardScaler()
X = scaler.fit_transform(X)
joblib.dump(scaler, "scaler.pkl")
X = X.reshape(X.shape[0], 24, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = build_lightweight_cnn(input_shape=(24,1))


# 2. Re-train with the weights
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,             # Increased epochs to give it time to adjust
    batch_size=16,
    verbose=1
)

y_pred = (model.predict(X_test) > 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
model.save("lightweight_cnn_model.h5")

Epoch 1/50


C:\Users\percy\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7988 - loss: 0.4247 - val_accuracy: 0.8359 - val_loss: 0.4171
Epoch 2/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9024 - loss: 0.2485 - val_accuracy: 0.7500 - val_loss: 0.4410
Epoch 3/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9204 - loss: 0.2216 - val_accuracy: 0.7695 - val_loss: 0.4683
Epoch 4/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9134 - loss: 0.2118 - val_accuracy: 0.8359 - val_loss: 0.3183
Epoch 5/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9247 - loss: 0.1830 - val_accuracy: 0.8164 - val_loss: 0.3901
Epoch 6/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9337 - loss: 0.1646 - val_accuracy: 0.8516 - val_loss: 0.3381
Epoch 7/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9357 - loss: 0.1788 - val_accuracy: 0.9023 - val_loss: 0.2100
Epoch 8/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9306 - loss: 0.1635 - val_accuracy: 0.7734 - val_loss: 0.5198
Epo

Confusion Matrix:
[[ 64   4]
 [  3 185]]


In [47]:
df = pd.read_csv("feature.csv")  # Change to pd.read_excel(file_name) if it's an Excel file
scaler = StandardScaler()
scaler = joblib.load("scaler.pkl")
df = scaler.transform(df)

# 2. Reshape to (Samples, 24, 1)
input_data = np.expand_dims(df, axis=-1)
print(f"Shape going into model: {input_data.shape}") 
# Should be (1, 24, 1) for a single file
# 3. Predict silently to avoid the Unicode crash
prediction = model.predict(input_data, verbose=0)

Shape going into model: (1, 24, 1)


c:\Users\percy\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [48]:
print(f"Prediction score: {prediction[0][0]:.4f}")

Prediction score: 0.0036


In [42]:
print("Class Distribution in y_train:")
print(pd.Series(X_train.flatten()).value_counts(normalize=True))

print("\nClass Distribution in y_test:")
print(pd.Series(y_test.flatten()).value_counts(normalize=True))

Class Distribution in y_train:
-1.164330    0.001551
-1.295863    0.001469
 0.874432    0.001347
 0.479833    0.001347
 0.348300    0.001102
               ...   
-0.689378    0.000041
 0.843915    0.000041
-1.208169    0.000041
-0.574484    0.000041
-0.435092    0.000041
Name: proportion, Length: 21307, dtype: float64

Class Distribution in y_test:
1.0    0.734375
0.0    0.265625
Name: proportion, dtype: float64
